In [2]:
import pandas as pd


import openmeteo_requests
import requests_cache
from retry_requests import retry

In [ ]:
df_provs = pd.read_excel("_Provincias_Info.xlsx", sheet_name="df_LatLon")
df_provs["lat"] = df_provs["Lat"].astype(float)
df_provs["lon"] = df_provs["Lon"].astype(float)
df_provs["prov"] = df_provs["Nombre"].astype(str)
df_provs["code"] = df_provs["CODIGO"].astype("Int8")
df_provs.drop(columns=["Lat", "Lon", "Nombre", "CODIGO"], inplace=True)
# df_provs.to_csv("Provinces_data.csv", index=False)
df_provs

,Capital,Comunidad autónoma,Nombre oficial,Poblacion,Superficie,%Poblacion,%Superficie2,lat,lon,prov,code
0,Madrid,Comunidad de Madrid,Madrid,7113886,8028,0.144802,0.015866,40.495,-3.7170,Madrid,28
1,Barcelona,Cataluña,Barcelona,5959941,7728,0.121314,0.015273,41.730,1.9837,Barcelona,8
2,Valencia,Comunitat Valenciana,Valencia/València,2763996,10807,0.056261,0.021358,39.370,-0.8000,Valencia,46
3,Alicante,Comunitat Valenciana,Alicante/Alacant,2033566,5817,0.041393,0.011496,38.478,-0.5680,Alicante,3
4,Sevilla,Andalucía,Sevilla,1977664,14036,0.040255,0.027740,37.435,-5.6820,Sevilla,41
5,Málaga,Andalucía,Málaga,1791183,7306,0.036459,0.014439,36.813,-4.7250,Málaga,29
6,Murcia,Región de Murcia,Murcia,1586989,11314,0.032303,0.022360,38.002,-1.4850,Murcia,30
7,Cádiz,Andalucía,Cádiz,1261420,7440,0.025676,0.014704,36.553,-5.7600,Cádiz,11
8,Palma,Illes Balears,"Balears, Illes",1249844,4992,0.025440,0.009866,39.574,2.9124,Islas Baleares,7
9,Las Palmas de Gran Canaria,Canarias,"Palmas, Las",1171547,4066,0.023847,0.008036,28.365,-14.5400,Las Palmas,35


In [4]:
num_init = 0
num_end = 10


# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
city_names = df_provs.iloc[num_init:num_end+1]["prov"].to_list()
params = {
	"latitude":df_provs.iloc[num_init:num_end+1]["lat"].to_list(),
	"longitude":df_provs.iloc[num_init:num_end+1]["lon"].to_list(),
	"start_date": "2020-01-01",
	"end_date": "2026-03-24",
	"daily": ["weather_code", "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "apparent_temperature_mean", "wind_speed_10m_max", "wind_gusts_10m_max", "shortwave_radiation_sum", "sunrise", "sunset", "daylight_duration", "precipitation_sum", "rain_sum", "snowfall_sum", "precipitation_hours", "surface_pressure_mean", "et0_fao_evapotranspiration_sum"],
}
responses = openmeteo.weather_api(url, params = params)

city_frames = []

# Process all locations and collect a dataframe per city
for city_name, response in zip(city_names, responses):
	print(f"\nCity: {city_name}")
	print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation: {response.Elevation()} m asl")
	print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
	# Process daily data. The order of variables needs to be the same as requested.
	daily = response.Daily()
	daily_weather_code = daily.Variables(0).ValuesAsNumpy()
	daily_temperature_2m_mean = daily.Variables(1).ValuesAsNumpy()
	daily_temperature_2m_max = daily.Variables(2).ValuesAsNumpy()
	daily_temperature_2m_min = daily.Variables(3).ValuesAsNumpy()
	daily_apparent_temperature_mean = daily.Variables(4).ValuesAsNumpy()
	daily_wind_speed_10m_max = daily.Variables(5).ValuesAsNumpy()
	daily_wind_gusts_10m_max = daily.Variables(6).ValuesAsNumpy()
	daily_shortwave_radiation_sum = daily.Variables(7).ValuesAsNumpy()
	daily_sunrise = daily.Variables(8).ValuesInt64AsNumpy()
	daily_sunset = daily.Variables(9).ValuesInt64AsNumpy()
	daily_daylight_duration = daily.Variables(10).ValuesAsNumpy()
	daily_precipitation_sum = daily.Variables(11).ValuesAsNumpy()
	daily_rain_sum = daily.Variables(12).ValuesAsNumpy()
	daily_snowfall_sum = daily.Variables(13).ValuesAsNumpy()
	daily_precipitation_hours = daily.Variables(14).ValuesAsNumpy()
	daily_surface_pressure_mean = daily.Variables(15).ValuesAsNumpy()
	daily_et0_fao_evapotranspiration_sum = daily.Variables(16).ValuesAsNumpy()
	
	daily_data = {"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	)}
	
	daily_data["weather_code"] = daily_weather_code
	daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
	daily_data["temperature_2m_max"] = daily_temperature_2m_max
	daily_data["temperature_2m_min"] = daily_temperature_2m_min
	daily_data["apparent_temperature_mean"] = daily_apparent_temperature_mean
	daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
	daily_data["wind_gusts_10m_max"] = daily_wind_gusts_10m_max
	daily_data["shortwave_radiation_sum"] = daily_shortwave_radiation_sum
	daily_data["sunrise"] = daily_sunrise
	daily_data["sunset"] = daily_sunset
	daily_data["daylight_duration"] = daily_daylight_duration
	daily_data["precipitation_sum"] = daily_precipitation_sum
	daily_data["rain_sum"] = daily_rain_sum
	daily_data["snowfall_sum"] = daily_snowfall_sum
	daily_data["precipitation_hours"] = daily_precipitation_hours
	daily_data["surface_pressure_mean"] = daily_surface_pressure_mean
	daily_data["et0_fao_evapotranspiration_sum"] = daily_et0_fao_evapotranspiration_sum
	
	daily_city_df = pd.DataFrame(data = daily_data)
	daily_city_df["city"] = city_name
	city_frames.append(daily_city_df)

daily_dataframe_2 = pd.concat(city_frames, ignore_index = True)
daily_dataframe_2


City: Madrid
Coordinates: 40.52724075317383°N -3.686431884765625°E
Elevation: 660.0 m asl
Timezone difference to GMT+0: 0s

City: Barcelona
Coordinates: 41.72231674194336°N 1.9536901712417603°E
Elevation: 567.0 m asl
Timezone difference to GMT+0: 0s

City: Valencia
Coordinates: 39.40245819091797°N -0.745849609375°E
Elevation: 346.0 m asl
Timezone difference to GMT+0: 0s

City: Alicante
Coordinates: 38.48857498168945°N -0.610565185546875°E
Elevation: 508.0 m asl
Timezone difference to GMT+0: 0s

City: Sevilla
Coordinates: 37.4340934753418°N -5.625°E
Elevation: 185.0 m asl
Timezone difference to GMT+0: 0s

City: Málaga
Coordinates: 36.8014030456543°N -4.730621337890625°E
Elevation: 370.0 m asl
Timezone difference to GMT+0: 0s

City: Murcia
Coordinates: 37.996482849121094°N -1.451629638671875°E
Elevation: 526.0 m asl
Timezone difference to GMT+0: 0s

City: Cádiz
Coordinates: 36.52021026611328°N -5.76470947265625°E
Elevation: 309.0 m asl
Timezone difference to GMT+0: 0s

City: Islas Balea

,date,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,apparent_temperature_mean,wind_speed_10m_max,wind_gusts_10m_max,shortwave_radiation_sum,sunrise,sunset,daylight_duration,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,surface_pressure_mean,et0_fao_evapotranspiration_sum,city
0,2020-01-01 00:00:00+00:00,2.0,3.147917,11.25,-2.30,0.386449,9.178235,11.520000,9.330000,1577864282,1577897875,33592.332031,0.0,0.0,0.0,0.0,952.290039,1.082871,Madrid
1,2020-01-02 00:00:00+00:00,3.0,2.483333,9.30,-2.50,-0.423794,8.534353,12.959999,9.230000,1577950689,1577984326,33636.273438,0.0,0.0,0.0,0.0,951.688293,1.020405,Madrid
2,2020-01-03 00:00:00+00:00,3.0,1.568750,6.60,-1.50,-1.070321,8.209263,12.959999,6.720000,1578037094,1578070778,33683.929688,0.0,0.0,0.0,0.0,953.317322,0.700336,Madrid
3,2020-01-04 00:00:00+00:00,1.0,3.754167,10.25,-1.25,0.239713,17.057314,38.160000,8.870000,1578123497,1578157232,33735.214844,0.0,0.0,0.0,0.0,954.555420,1.079652,Madrid
4,2020-01-05 00:00:00+00:00,0.0,3.179167,11.45,-2.35,0.025346,9.931042,19.440001,9.410000,1578209897,1578243688,33790.031250,0.0,0.0,0.0,0.0,950.722839,1.111593,Madrid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25020,2026-03-20 00:00:00+00:00,3.0,12.800000,19.50,7.05,10.342389,14.003029,32.760002,19.309999,1773987254,1774030989,43733.839844,0.0,0.0,0.0,0.0,1002.611389,3.358055,Vizcaya
25021,2026-03-21 00:00:00+00:00,3.0,10.904166,17.90,5.00,8.982385,10.993271,27.719999,19.680000,1774073548,1774117460,43911.101562,0.0,0.0,0.0,0.0,1002.135193,3.047693,Vizcaya
25022,2026-03-22 00:00:00+00:00,51.0,9.250001,15.25,3.10,7.451178,17.782686,41.399998,18.990000,1774159843,1774203932,44087.492188,0.4,0.4,0.0,4.0,1004.717285,2.488391,Vizcaya
25023,2026-03-23 00:00:00+00:00,51.0,9.522917,14.35,4.85,7.798176,11.341428,28.440001,16.090000,1774246138,1774290403,44263.125000,0.9,0.9,0.0,5.0,1010.509277,2.198060,Vizcaya


In [5]:
extract_date = pd.to_datetime("today")
place = "top0-10_cities"
dates_from_to = "2020-2026"
csv_title = f"daily_{place}_{dates_from_to}_{extract_date.strftime('%Y-%m-%d')}_extracted.csv"

daily_dataframe_2.to_csv(csv_title, index = False)
daily_dataframe_2

,date,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,apparent_temperature_mean,wind_speed_10m_max,wind_gusts_10m_max,shortwave_radiation_sum,sunrise,sunset,daylight_duration,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,surface_pressure_mean,et0_fao_evapotranspiration_sum,city
0,2020-01-01 00:00:00+00:00,2.0,3.147917,11.25,-2.30,0.386449,9.178235,11.520000,9.330000,1577864282,1577897875,33592.332031,0.0,0.0,0.0,0.0,952.290039,1.082871,Madrid
1,2020-01-02 00:00:00+00:00,3.0,2.483333,9.30,-2.50,-0.423794,8.534353,12.959999,9.230000,1577950689,1577984326,33636.273438,0.0,0.0,0.0,0.0,951.688293,1.020405,Madrid
2,2020-01-03 00:00:00+00:00,3.0,1.568750,6.60,-1.50,-1.070321,8.209263,12.959999,6.720000,1578037094,1578070778,33683.929688,0.0,0.0,0.0,0.0,953.317322,0.700336,Madrid
3,2020-01-04 00:00:00+00:00,1.0,3.754167,10.25,-1.25,0.239713,17.057314,38.160000,8.870000,1578123497,1578157232,33735.214844,0.0,0.0,0.0,0.0,954.555420,1.079652,Madrid
4,2020-01-05 00:00:00+00:00,0.0,3.179167,11.45,-2.35,0.025346,9.931042,19.440001,9.410000,1578209897,1578243688,33790.031250,0.0,0.0,0.0,0.0,950.722839,1.111593,Madrid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25020,2026-03-20 00:00:00+00:00,3.0,12.800000,19.50,7.05,10.342389,14.003029,32.760002,19.309999,1773987254,1774030989,43733.839844,0.0,0.0,0.0,0.0,1002.611389,3.358055,Vizcaya
25021,2026-03-21 00:00:00+00:00,3.0,10.904166,17.90,5.00,8.982385,10.993271,27.719999,19.680000,1774073548,1774117460,43911.101562,0.0,0.0,0.0,0.0,1002.135193,3.047693,Vizcaya
25022,2026-03-22 00:00:00+00:00,51.0,9.250001,15.25,3.10,7.451178,17.782686,41.399998,18.990000,1774159843,1774203932,44087.492188,0.4,0.4,0.0,4.0,1004.717285,2.488391,Vizcaya
25023,2026-03-23 00:00:00+00:00,51.0,9.522917,14.35,4.85,7.798176,11.341428,28.440001,16.090000,1774246138,1774290403,44263.125000,0.9,0.9,0.0,5.0,1010.509277,2.198060,Vizcaya
